In [1]:
import os
import glob 
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models 

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch: 2.10.0+cu128
CUDA available: True


In [2]:
DATA_DIR = "/kaggle/input/datasets/mishbhaul/pre-processed-indian-folk-music"
OUTPUT_DIR = "/kaggle/working/"

In [3]:
TARGET_FRAMES = 128     
IMG_SIZE = 224

EPOCHS = 5
BATCH_SIZE = 16
LR = 1e-4
VAL_SPLIT = 0.2
FREEZE_BACKBONE = False  

In [4]:
def load_style_folders_with_songs(data_dir):
    """
    Load every class folder's *_melspec.npy array plus its song labels.

    Expected structure:
    DATA_DIR/
        Bauls/
            Bauls/
                Bauls_melspec.npy
                Bauls.csv
        Bhajans/
            Bhajans/
                Bhajans_melspec.npy
                Bhajans.csv
        ...
    """

    class_names = sorted(
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    )

    if not class_names:
        raise RuntimeError(f"No class folders found in {data_dir}")

    X, y, songs = [], [], []

    for label, cls in tqdm(
    enumerate(class_names),
    total=len(class_names),
    desc="Loading classes"
):

        # Outer class folder
        outer_dir = os.path.join(data_dir, cls)

        # Inner folder (same name as class)
        cls_dir = os.path.join(outer_dir, cls)

        if not os.path.isdir(cls_dir):
            print(f"[WARN] Missing inner folder: {cls_dir}")
            continue

        # Find spectrogram
        npy_files = glob.glob(os.path.join(cls_dir, "*_melspec.npy"))

        if not npy_files:
            print(f"[WARN] No *_melspec.npy found in {cls_dir}")
            continue

        arr = np.load(npy_files[0])

        if arr.ndim == 2:
            arr = arr[np.newaxis, ...]
        elif arr.ndim != 3:
            raise ValueError(f"Unexpected shape {arr.shape}")

        # Find CSV
        csv_files = glob.glob(os.path.join(cls_dir, "*.csv"))

        if csv_files:
            meta = pd.read_csv(csv_files[0])

            if "song" in meta.columns:
                song_col = meta["song"].astype(str).tolist()
            else:
                print(f"[WARN] 'song' column missing in {csv_files[0]}")
                song_col = [f"{cls}_{i}" for i in range(len(arr))]
        else:
            print(f"[WARN] No CSV found in {cls_dir}")
            song_col = [f"{cls}_{i}" for i in range(len(arr))]

        if len(song_col) != len(arr):
            print(f"[WARN] {cls}: {len(song_col)} metadata rows vs {len(arr)} samples")

        for i, sample in enumerate(arr):
            X.append(sample.astype(np.float32))
            y.append(label)
            songs.append(song_col[i] if i < len(song_col) else f"{cls}_{i}")

    print(f"Loaded {len(X)} samples across {len(class_names)} classes.")
    print("Classes:", class_names)

    return X, y, songs, class_names


# Load dataset
X, y, songs, CLASS_NAMES = load_style_folders_with_songs(DATA_DIR)
NUM_CLASSES = len(CLASS_NAMES)

print("Example sample shape:", X[0].shape)

Loading classes:   0%|          | 0/15 [00:00<?, ?it/s]

Loaded 385543 samples across 15 classes.
Classes: ['Bauls', 'Bhatiali', 'Bhavageethe', 'Bihu', 'Garba', 'Gidha', 'Kajri', 'Lavani', 'Maand', 'Naatupura_Paatu', 'Sohar', 'Sufi', 'Tamang_selo', 'UttaraKhandi', 'Veeragase']
Example sample shape: (128, 130)


In [5]:
def standardize(spec):
    return (spec - spec.mean()) / (spec.std() + 1e-6)

def fix_length(spec, target_frames=TARGET_FRAMES):
    n_mels, n_frames = spec.shape
    if n_frames == target_frames:
        return spec
    if n_frames > target_frames:
        start = (n_frames - target_frames) // 2
        return spec[:, start:start + target_frames]
    pad_total = target_frames - n_frames
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return np.pad(spec, ((0, 0), (pad_left, pad_right)), mode="constant")

def spec_augment(spec):
    spec = spec.copy()
    n_mels, n_frames = spec.shape

    f0 = np.random.randint(0, max(1, n_mels - 10))
    spec[f0:f0 + 10, :] = 0.0   # frequency mask

    t0 = np.random.randint(0, max(1, n_frames - 15))
    spec[:, t0:t0 + 15] = 0.0   # time mask

    return spec

In [14]:
import torch
from torch.utils.data import Dataset

class MelSpecDatasetAST(Dataset):
    """
    Takes a RAW mel-spectrogram (shape: n_mels x time_frames, e.g. 128 x 130)
    and prepares it exactly the way AST needs it.
    """
    def __init__(self, X, y, train=True):
        self.X = X          # list of raw spectrograms
        self.y = y          # list of integer labels
        self.train = train  # True = training set (apply augmentation)
                             # False = validation set (no augmentation)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        spec = self.X[idx]                       # shape: (n_mels, time_frames)

        spec = fix_length(spec, TARGET_FRAMES)    # make every clip the same width
        spec = standardize(spec)                  # zero mean, unit variance

        if self.train:
            spec = spec_augment(spec)             # only augment training data

        spec = torch.tensor(spec, dtype=torch.float32)  # numpy -> tensor
        spec = spec.T                                    # (n_mels, time) -> (time, n_mels)
                                                          # AST wants time FIRST

        label = self.y[idx]
        return spec, label

In [15]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=VAL_SPLIT, random_state=SEED)
train_idx, val_idx = next(gss.split(X, y, groups=songs))

X_train = [X[i] for i in train_idx]
y_train = [y[i] for i in train_idx]
X_val   = [X[i] for i in val_idx]
y_val   = [y[i] for i in val_idx]

print(f"Train clips: {len(X_train)} | Val clips: {len(X_val)}")

Train clips: 310988 | Val clips: 74555


In [16]:
train_dataset = MelSpecDatasetAST(X_train, y_train, train=True)
val_dataset   = MelSpecDatasetAST(X_val,   y_val,   train=False)

# sanity check — this should now print a clean 2D shape, no errors
sample_spec, sample_label = train_dataset[0]
print("Shape AST will see:", sample_spec.shape)

MAX_LENGTH, NUM_MEL_BINS = sample_spec.shape
print(f"AST will see each clip as: {MAX_LENGTH} time steps x {NUM_MEL_BINS} mel bins")
print(f"Number of classes: {NUM_CLASSES}")

Shape AST will see: torch.Size([128, 128])
AST will see each clip as: 128 time steps x 128 mel bins
Number of classes: 15


In [17]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 🧠 Audio Spectrogram Transformer (AST)

Instead of listening to raw audio, **Audio Spectrogram Transformer (AST)** *looks at sound*. It treats each **mel-spectrogram** as an image, splits it into small patches, and converts each patch into a token. These tokens are processed by a **Transformer encoder**, which uses **self-attention** to learn relationships between different time-frequency regions of the spectrogram.

A special **[CLS] token** summarizes the entire audio clip, and its final representation is passed to a classification layer to predict one of the **15 Bangla music genres**.

Since AST is **pretrained on AudioSet (2M+ labeled audio clips)**, it already understands a wide range of acoustic patterns. Fine-tuning this pretrained model on our dataset enables it to effectively distinguish between different Bangla music genres while leveraging knowledge learned from large-scale audio data.

### 🔍 AST Pipeline

```text

Mel-Spectrogram
   │
   ▼
Split into Image Patches
   │
   ▼
Patch Embeddings + [CLS] Token
   │
   ▼
Transformer Encoder (Self-Attention)
   │
   ▼
[CLS] Representation
   │
   ▼
Linear Classification Head
   │
   ▼
15 Bangla Music Genres
```

# 1. Installs & Imports

In [19]:
!pip install transformers torchaudio --quiet

from transformers import ASTForAudioClassification, ASTConfig

**MelSpecDataset** : (n_mels, time_frames) - the librosa default but
**AST expects** : (time_frames, num_mel_bins)

# 2. ASTDataset

In [22]:
PRETRAINED_MODEL = "MIT/ast-finetuned-audioset-10-10-0.4593"

config = ASTConfig.from_pretrained(
    PRETRAINED_MODEL,
    num_labels=NUM_CLASSES,
    max_length=MAX_LENGTH,
    num_mel_bins=NUM_MEL_BINS,
)

model = ASTForAudioClassification.from_pretrained(
    PRETRAINED_MODEL,
    config=config,
    ignore_mismatched_sizes=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                                                          | Status   |                                                                                                
-------------------------------------------------------------+----------+------------------------------------------------------------------------------------------------
audio_spectrogram_transformer.embeddings.position_embeddings | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1, 1214, 768]) vs model:torch.Size([1, 146, 768])
classifier.dense.weight                                      | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([15, 768])        
classifier.dense.bias                                        | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([15])                  

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not mat

ASTForAudioClassification(
  (audio_spectrogram_transformer): ASTModel(
    (embeddings): ASTEmbeddings(
      (patch_embeddings): ASTPatchEmbeddings(
        (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ASTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ASTLayer(
          (attention): ASTAttention(
            (attention): ASTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ASTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ASTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=T

In [23]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for specs, labels in tqdm(loader, desc="Training", leave=False):
        specs, labels = specs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(input_values=specs).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * specs.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    for specs, labels in tqdm(loader, desc="Validating", leave=False):
        specs, labels = specs.to(device), labels.to(device)
        logits = model(input_values=specs).logits
        loss = criterion(logits, labels)

        total_loss += loss.item() * specs.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [25]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

EPOCHS = 1
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)  # small LR — we're fine-tuning, not training from scratch
criterion = nn.CrossEntropyLoss()

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

# outer tqdm bar = tracks progress across EPOCHS (not batches)
for epoch in tqdm(range(1, EPOCHS + 1), desc="Overall Progress"):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_ast_model.pt")
        print(f"  -> new best model saved (val_acc={val_acc:.4f})")

print(f"\nDone. Best validation accuracy: {best_val_acc:.4f}")

Overall Progress:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/19437 [00:00<?, ?it/s]

Validating:   0%|          | 0/4660 [00:00<?, ?it/s]

Epoch 1/1 | Train Loss: 0.2060, Train Acc: 0.9372 | Val Loss: 1.4293, Val Acc: 0.7141
  -> new best model saved (val_acc=0.7141)

Done. Best validation accuracy: 0.7141
